# NB02 - Data Transformation

Input: raw JSON files under `data/raw/tmdb/` (produced by NB01).
Output: tidy CSV tables under `data/processed/`, ready for analysis in NB03.

This notebook does not call the API. It reads the raw responses saved by NB01 and turns them into two clean tables:

1. `movies.csv` - one row per movie, with the columns the analysis needs (year, decade, runtime, rating, votes, language, budget, revenue).
2. `movie_genres.csv` - one row per movie-genre pair (a "long" table). A movie can have several genres, so we `explode` the genre list into separate rows. This shape makes it easy to count genres per decade in NB03.



## 1. Imports and paths

In [1]:
import json
from pathlib import Path

import pandas as pd

RAW_DIR = Path("..") / "data" / "raw" / "tmdb"
DETAILS_DIR = RAW_DIR / "details"
PROCESSED_DIR = Path("..") / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

detail_files = sorted(DETAILS_DIR.glob("*.json"))
print(f"Found {len(detail_files)} movie detail files to process.")
if not detail_files:
    raise RuntimeError("No detail files found. Run NB01 first to collect the data.")



Found 1220 movie detail files to process.


## 2. Genre lookup

The movie records store genres as `{"id": 28, "name": "Action"}`. We also load the standalone `genres.json` as a reference, but the names inside each movie record are enough for our table.

In [2]:
with open(RAW_DIR / "genres.json", encoding="utf-8") as f:
    genres_raw = json.load(f)

genre_lookup = {g["id"]: g["name"] for g in genres_raw.get("genres", [])}
print(f"{len(genre_lookup)} genres available, e.g.:",
      dict(list(genre_lookup.items())[:5]))


19 genres available, e.g.: {28: 'Action', 12: 'Adventure', 16: 'Animation', 35: 'Comedy', 80: 'Crime'}


## 3. Build the movie-level table

We read each raw movie record and keep only the fields we need. A few cleaning decisions, each with a reason:

- `runtime`: TMDB uses `0` when the runtime is unknown. A 0-minute film is not real, so we convert 0 to missing (`NaN`) to avoid dragging the averages down.
- `budget` / `revenue`: same idea - `0` almost always means "not reported", so we treat it as missing.
- `year`: taken from `release_date`. Records without a valid date are dropped, since the whole project is organised by year.
- `decade`: derived from the year (e.g. 1994 -> 1990) so NB03 can group easily.

In [3]:
def parse_movie(record):
    """Turn one raw TMDB movie record into a flat dictionary of the fields we need."""
    release_date = record.get("release_date") or ""
    year = int(release_date[:4]) if release_date[:4].isdigit() else None

    runtime = record.get("runtime") or 0
    budget = record.get("budget") or 0
    revenue = record.get("revenue") or 0

    genre_names = [g["name"] for g in record.get("genres", [])]

    return {
        "movie_id": record.get("id"),
        "title": record.get("title"),
        "year": year,
        "runtime": runtime if runtime > 0 else None,      # 0 means unknown
        "vote_average": record.get("vote_average"),
        "vote_count": record.get("vote_count"),
        "budget": budget if budget > 0 else None,
        "revenue": revenue if revenue > 0 else None,
        "original_language": record.get("original_language"),
        "genres": genre_names,                            # list, expanded later
    }

rows = []
for path in detail_files:
    record = json.loads(path.read_text(encoding="utf-8"))
    rows.append(parse_movie(record))

movies = pd.DataFrame(rows)

# Drop movies without a usable year, then add the decade column
movies = movies.dropna(subset=["year"]).copy()
movies["year"] = movies["year"].astype(int)
movies["decade"] = (movies["year"] // 10) * 10

print(f"Movie table: {len(movies)} rows, {movies['year'].min()}-{movies['year'].max()}")
movies.head()


Movie table: 1220 rows, 1960-2020


,movie_id,title,year,runtime,vote_average,vote_count,budget,revenue,original_language,genres,decade
0,100,"Lock, Stock and Two Smoking Barrels",1998,105,8.097,7279,1350000.0,28356188.0,en,"[Comedy, Crime]",1990
1,10014,A Nightmare on Elm Street Part 2: Freddy's Rev...,1985,87,5.800,2141,3000000.0,30000121.0,en,[Horror],1980
2,10020,Beauty and the Beast,1991,84,7.726,10772,25000000.0,424967620.0,en,"[Romance, Family, Animation, Fantasy]",1990
3,100402,Captain America: The Winter Soldier,2014,136,7.655,20180,170000000.0,714766572.0,en,"[Action, Adventure, Science Fiction]",2010
4,10072,A Nightmare on Elm Street 3: Dream Warriors,1987,96,6.717,2065,4450000.0,44793222.0,en,"[Horror, Thriller, Fantasy]",1980
